# LECTURE 21 — HANDS-ON EXERCISE
### MODULE 9: INTERPRETABILITY, GOVERNANCE & COMMUNICATION

**HANDS-ON EXERCISE** — Prepare a Non-Technical Policy Brief Explaining an ML Model

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from matplotlib.patches import Patch

import xgboost as xgb
import shap

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: BANKING CRISIS DATA (provided)

In [ ]:
def generate_banking_crisis_data(n_obs: int = 700, seed: int = 42) -> pd.DataFrame:
    """
    700 country-year observations for banking crisis classification.

    Features:
      npl_ratio, capital_adequacy, liquidity_ratio, roa,
      credit_growth, inflation, gdp_growth, current_account,
      foreign_reserves, exchange_rate_vol
    Target: banking_crisis (1 = crisis, 0 = no crisis), ~15 % positive rate.
    """
    rng = np.random.default_rng(seed)

    npl_ratio         = rng.gamma(2.5, 2.5, n_obs)
    capital_adequacy  = rng.normal(13, 3, n_obs).clip(4, 30)
    liquidity_ratio   = rng.normal(30, 8, n_obs).clip(10, 70)
    roa               = rng.normal(1.5, 1.2, n_obs)
    credit_growth     = rng.normal(12, 8, n_obs)
    inflation         = rng.gamma(2, 3.5, n_obs)
    gdp_growth        = rng.normal(3.5, 2, n_obs)
    current_account   = rng.normal(-2, 4, n_obs)
    foreign_reserves  = rng.gamma(3, 1.5, n_obs)
    exchange_rate_vol = rng.gamma(1.5, 2, n_obs)

    logit = (
        -4.5
        + 0.28 * npl_ratio
        - 0.16 * capital_adequacy
        - 0.05 * liquidity_ratio
        - 0.38 * roa
        + 0.09 * credit_growth
        + 0.10 * inflation
        - 0.22 * gdp_growth
        - 0.08 * current_account
        - 0.15 * foreign_reserves
        + 0.12 * exchange_rate_vol
        + rng.normal(0, 0.5, n_obs)
    )
    prob   = 1 / (1 + np.exp(-logit))
    crisis = (rng.uniform(0, 1, n_obs) < prob).astype(int)

    return pd.DataFrame({
        'npl_ratio':         npl_ratio,
        'capital_adequacy':  capital_adequacy,
        'liquidity_ratio':   liquidity_ratio,
        'roa':               roa,
        'credit_growth':     credit_growth,
        'inflation':         inflation,
        'gdp_growth':        gdp_growth,
        'current_account':   current_account,
        'foreign_reserves':  foreign_reserves,
        'exchange_rate_vol': exchange_rate_vol,
        'banking_crisis':    crisis,
    })


df = generate_banking_crisis_data()
FEATURE_COLS = [c for c in df.columns if c != 'banking_crisis']
TARGET_COL   = 'banking_crisis'

print(f"Dataset : {df.shape[0]} obs  x  {df.shape[1]} variables")
vc = df[TARGET_COL].value_counts()
print(f"Class distribution — No Crisis: {vc[0]}  |  Crisis: {vc[1]}")
print(df.describe().round(2).to_string())

## ── SECTION 3: PREPROCESSING (provided)

In [ ]:
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y,
)

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = float(n_neg) / float(n_pos)   # XGBoost imbalance parameter

print(f"\nTrain : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

## ── SECTION 4: TRAIN XGBOOST  ◄─ YOUR TASK (TODO 1)

Train an XGBoost classifier using XGBClassifier.

Suggested hyperparameters:

n_estimators=300, max_depth=4, learning_rate=0.05,

subsample=0.80, colsample_bytree=0.80,

scale_pos_weight=scale_pos_weight,   ← handles class imbalance

use_label_encoder=False,

eval_metric='auc',

random_state=SEED, verbosity=0

After training:

a) Get predicted probabilities: y_prob = model.predict_proba(X_test)[:, 1]

b) Get class predictions:       y_pred = (y_prob >= 0.40).astype(int)

(threshold 0.40 is often better than 0.50 for imbalanced crisis data)

c) Print classification_report(y_test, y_pred,

target_names=['No Crisis', 'Crisis'])

d) Print AUC: roc_auc_score(y_test, y_prob)

In [ ]:
# WRITE YOUR CODE BELOW
# model = xgb.XGBClassifier(...)
# model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
# y_prob = ...
# y_pred = ...
# print(classification_report(...))

## ── SECTION 5: SHAP GLOBAL IMPORTANCE  ◄─ YOUR TASK (TODO 2)

1. Create SHAP explainer:

explainer   = shap.TreeExplainer(model)

shap_values = explainer(pd.DataFrame(X_test, columns=FEATURE_COLS))

2. Plot SHAP bar chart (global feature importance):

fig, ax = plt.subplots(figsize=(8, 5))

shap.plots.bar(shap_values, max_display=10, ax=ax, show=False)

ax.set_title('Global Feature Importance — Banking Crisis Model')

plt.tight_layout(); plt.show()

3. Plot SHAP beeswarm plot:

fig, ax = plt.subplots(figsize=(9, 6))

shap.plots.beeswarm(shap_values, max_display=10, ax=ax, show=False)

plt.tight_layout(); plt.show()

In [ ]:
# WRITE YOUR CODE BELOW
# explainer   = shap.TreeExplainer(model)
# shap_values = explainer(pd.DataFrame(X_test, columns=FEATURE_COLS))
# ...

## ── SECTION 6: SHAP LOCAL EXPLANATION  ◄─ YOUR TASK (TODO 3)

Select ONE country-year observation from X_test that your model predicts

as HIGH risk (y_prob >= 0.70).

1. Find its index:

high_risk_idx = np.where(y_prob >= 0.70)[0][0]

2. Plot a waterfall chart:

fig, ax = plt.subplots(figsize=(9, 5))

shap.plots.waterfall(shap_values[high_risk_idx], ax=ax, show=False)

ax.set_title(f'Why does the model flag this observation as high risk?  '

f'(Predicted prob = {y_prob[high_risk_idx]:.2f})')

plt.tight_layout(); plt.show()

3. In 2–3 sentences, explain in plain language what the waterfall chart shows.

In [ ]:
# WRITE YOUR CODE BELOW
# high_risk_idx = np.where(y_prob >= 0.70)[0][0]
# ...

# Your plain-language explanation:
# "This observation was flagged as high risk because ..."

## ── SECTION 7: POLICY COMMUNICATION CHART  ◄─ YOUR TASK (TODO 4)

Create a clean, non-technical horizontal bar chart showing the average

SHAP impact of each feature.  Bars should be coloured:

RED   if the feature INCREASES crisis probability on average

GREEN if the feature DECREASES crisis probability on average

Steps:

mean_shap_vals = pd.Series(

np.abs(shap_values.values).mean(axis=0),

index=FEATURE_COLS

).sort_values(ascending=True)

Determine direction for each feature:

direction = []

for feat in mean_shap_vals.index:

fi   = FEATURE_COLS.index(feat)

corr = np.corrcoef(X_test[:, fi], shap_values.values[:, fi])[0, 1]

direction.append('raises_risk' if corr > 0 else 'lowers_risk')

bar_colors = ['#F44336' if d == 'raises_risk' else '#4CAF50' for d in direction]

Title the chart: 'Key Drivers of Banking Crisis Risk'

Add a legend explaining red = raises risk, green = lowers risk.

In [ ]:
# WRITE YOUR CODE BELOW
# ...

## ── SECTION 8: WRITE A POLICY BRIEF NARRATIVE  ◄─ YOUR TASK (TODO 5)

Fill in the policy brief template below based on your results.

Replace every [...] with your actual findings from Sections 4–7.

This output is what a central bank economist might include in a supervisory

report — keep it clear, jargon-free, and action-oriented.

In [ ]:
policy_brief_template = """
╔══════════════════════════════════════════════════════════════════════╗
║  POLICY BRIEF — Banking Crisis Early Warning System                  ║
║  Prepared for: Central Bank Risk Division                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  MODEL PERFORMANCE                                                   ║
║  The XGBoost model achieved an AUC of [...] on the test set,         ║
║  correctly identifying [...]% of actual crises.                      ║
║                                                                      ║
║  KEY RISK DRIVERS (top 3 from SHAP analysis)                        ║
║  1. [...] — [explain the direction and magnitude of its effect]      ║
║  2. [...] — [explain the direction and magnitude of its effect]      ║
║  3. [...] — [explain the direction and magnitude of its effect]      ║
║                                                                      ║
║  CASE STUDY: HIGH-RISK OBSERVATION                                   ║
║  The model flagged one observation with a crisis probability of [...]  ║
║  The main contributing factors were: [list 2–3 from waterfall chart] ║
║                                                                      ║
║  POLICY RECOMMENDATION                                               ║
║  Based on these findings, the central bank should monitor [...]      ║
║  most closely. Early warning triggers should be set at:              ║
║    - NPL ratio above [...] %                                         ║
║    - Capital adequacy below [...] %                                  ║
║    - Credit growth exceeding [...] %                                 ║
║                                                                      ║
║  LIMITATIONS & GOVERNANCE                                            ║
║  This model is a decision-support tool, not a substitute for         ║
║  supervisory judgement. Predictions should be reviewed quarterly      ║
║  and the model re-trained annually with updated data.                ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(policy_brief_template)

## ── SECTION 9: DISCUSSION QUESTIONS

In [ ]:
print("""
── Discussion Questions ───────────────────────────────────────────────────────
 1. The model uses a decision threshold of 0.40 rather than 0.50.
    What are the consequences of lowering the threshold for a central bank?
    (Think: false alarms vs. missed crises.)

 2. A board member asks: "How do we know the model isn't just lucky?"
    How would you use cross-validation and bootstrap CIs to respond?

 3. The top SHAP feature explains the most variance. However, could it be
    an effect rather than a cause of crisis? How does this affect policy?

 4. SHAP values tell you about correlation / influence, not causality.
    What additional analysis would you need before acting on the findings?

 5. Describe two governance actions your institution should take before
    deploying this model in a real supervisory context.
──────────────────────────────────────────────────────────────────────────────
""")

print("=== EXERCISE COMPLETE ===")